In [1]:
import hoda
import tensorly as tl

print(tl.get_backend())
%pip freeze | grep moabb

cupy
moabb==0.4.6
Note: you may need to restart the kernel to use updated packages.


In [2]:
from moabb.paradigms import P300
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation

tmin = 0
tmax=0.8
fmin=0.5
fmax = 16
sfreq = 32

paradigm = P300(resample=sfreq, tmin=tmin, tmax=tmax, fmin=fmin, fmax=fmax)
datasets = [
       #BI2012(),
       #BI2013a(),
       #BI2014a(),
       #BI2014b(),
       #BI2015a(),
       #BI2015b(),
       BNCI2014008(),
       BNCI2014009(),
       #BNCI2015_003(),
       #Cattan2019_VR(),
       #EPFLP300(),
       #Huebner2017(),
       #Huebner2018(),
       #Lee2019_ERP(),
       #Sosulski2019()   
   ]



In [3]:
from sklearn.pipeline import Pipeline
from mne.decoding import Scaler
from hoda.hoda import  BTTDA
from classification import Vectorize
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold

pipeline = Pipeline([
    ('scaler1', Scaler(scalings='mean', with_mean=False)),
    ('bttda', BTTDA(
        n_blocks=32,
        hoda_params=dict(
            max_iter=64,
            rank=None,
            tol=1e-8,
            init ='svd',
            shrinkage='lw',
            toeplitz=None,
            obj='rt',
            solver='lanczos',        
            verbose=False,
            taper=False,
            lasso=False,
            prune=True,
            keep_train_info=False
        ),
        keep_train_info=False,
        deflate_transform=False,
        verbose=False,
    )),
    ('vec', Vectorize()),
    ('scaler2', StandardScaler()),
    ('lda',LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr'))
])

cv = StratifiedKFold(n_splits=5)

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

def evaluate_fold(dataset, run, X, labels, fold, train_idc, test_idc):
    train_idc = run_idc[train_idc]
    test_idc = run_idc[test_idc]
    pipeline[:2].fit(X[train_idc], labels[train_idc])
    results = []
    for n_blocks in range(1, pipeline[1].n_blocks+1): 
        Xt_train = pipeline[:1].transform(X[train_idc])
        Xt_train = pipeline[1].transform(X[train_idc], n_blocks=n_blocks)
        Xt_test = pipeline[:1].transform(X[test_idc])
        Xt_test = pipeline[1].transform(X[test_idc], n_blocks=n_blocks)
        pipeline[2:].fit(Xt_train, labels[train_idc])
        pred = pipeline[2:].decision_function(Xt_test)
        score = roc_auc_score(labels[test_idc], pred)
        results.append(dict(
            dataset=dataset.code,
            subject=run[0],
            session=run[1],
            run=run[2],
            fold=fold,
            n_blocks=n_blocks,
            roc_auc_score=score
        ))
    return results

df = []
for dataset in datasets:
    X, labels, meta = paradigm.get_data(dataset, subjects=[1])
    idc = meta.reset_index()\
        .groupby(['subject', 'session', 'run'])\
        .index.aggregate(list)
    for run, run_idc in idc.items():
        print(dataset.code + ' ' + str(run))
        run_idc = np.array(run_idc)    
        fold_results = Parallel(n_jobs=cv.n_splits)\
            (delayed(evaluate_fold)(dataset, run, X, labels, fold, train_idc, test_idc)\
             for fold, (train_idc, test_idc) in enumerate(cv.split(X[run_idc], labels[run_idc])))
        df += fold_results


df = pd.DataFrame([item for sublist in df for item in sublist])

008-2014 (1, 'session_0', 'run_0')


/usr/local/lib/python3.10/dist-packages/moabb/paradigms/p300.py:181: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  X.append(dataset.unit_factor * epochs.get_data())


In [ ]:
df.to_csv('results_blocks_sort.csv')
df

In [ ]:
import seaborn as sns

df_agg = df.groupby(['dataset', 'subject', 'session', 'run', 'n_blocks'])
df_agg = df_agg.roc_auc_score.aggregate('mean')
df_agg = df_agg.reset_index()
sns.relplot(
    data=df_agg,
    col='dataset',
    #col_wrap=4,
    x='n_blocks',
    y='roc_auc_score',
    hue='session',
    kind="line",
    #units='subject',
    #estimator=None,
)